# 01 Exploratory data analysis: LCF 2013
Run top to bottom. Figures are saved to `reports/figures/` as PDF (vector) for the thesis.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.config import FIG_DIR, TARGET, FEATURES_A, FEATURES_B, CATEGORICAL
from src.data import load_clean, add_segments
sns.set_theme(style='whitegrid')
df = add_segments(load_clean())
df.shape

## 1. Structure and quality

In [ ]:
display(df.describe().T)
print('missing:', df.isna().sum().sum())
print('top-coded expenditure:', (df[TARGET] >= 1175).sum(), '| top-coded income:', (df['income'] >= df['income'].max() - 0.01).sum(), '| zero income:', (df['income'] == 0).sum())

## 2. Target distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(df[TARGET], bins=40, ax=ax[0]); ax[0].set_xlabel('Weekly expenditure (GBP)')
sns.histplot(np.log(df[TARGET]), bins=40, ax=ax[1]); ax[1].set_xlabel('log(expenditure)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'target_distribution.pdf'); plt.show()
print('skew raw:', round(df[TARGET].skew(), 2), '| skew log:', round(np.log(df[TARGET]).skew(), 2))

## 3. Outliers (IQR rule)

In [ ]:
q1, q3 = df[TARGET].quantile([.25, .75]); iqr = q3 - q1
print('IQR upper fence:', round(q3 + 1.5 * iqr, 1), '| rows above:', int((df[TARGET] > q3 + 1.5 * iqr).sum()))

## 4. Expenditure by categorical predictor

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9)); axes = axes.ravel()
for a, c in zip(axes, CATEGORICAL):
    sns.boxplot(data=df, x=c, y=TARGET, ax=a, showfliers=False); a.set_title(c)
for a in axes[len(CATEGORICAL):]: a.axis('off')
plt.tight_layout(); plt.savefig(FIG_DIR / 'expenditure_by_category.pdf'); plt.show()

## 5. Income vs expenditure

In [ ]:
plt.figure(figsize=(6, 4)); sns.scatterplot(data=df, x='income', y=TARGET, alpha=.25, s=12)
plt.savefig(FIG_DIR / 'income_vs_expenditure.pdf'); plt.show()
print('Pearson r:', round(df[['income', TARGET]].corr().iloc[0, 1], 3))

## 6. Correlation matrix (codes as numbers, indicative only)

In [ ]:
plt.figure(figsize=(9, 7)); sns.heatmap(df[FEATURES_B + [TARGET]].corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.tight_layout(); plt.savefig(FIG_DIR / 'correlation_matrix.pdf'); plt.show()

## 7. Segments and weights

In [ ]:
print(df['segment'].value_counts())
print(df.groupby('segment', observed=True)[TARGET].agg(['min', 'max', 'mean']).round(1))
w = pd.read_stata('../data/raw/icfforworkbook.dta', convert_categoricals=False)['weightar']
print(w.describe().round(3))